# RSNA Knee MRI — Huấn luyện trên Colab (A100 80GB)

Text-guided knowledge distillation: **huấn luyện** bằng ảnh + report,
**inference** chỉ cần ảnh + metadata series.

Notebook này chỉ *gọi* các hàm trong package `knee_mri`; toàn bộ logic nằm trong
`src/` nên chạy được cả ngoài notebook và có test bao phủ.


In [ ]:
# 1. Mount Drive và đưa package vào path
import os, sys
from google.colab import drive

drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/Addressing_challenge_RNSA_KNEE'
assert os.path.isdir(PROJECT), f'Không tìm thấy project: {PROJECT}'
sys.path.insert(0, os.path.join(PROJECT, 'src'))
os.chdir(PROJECT)
print('Thư mục làm việc:', os.getcwd())


In [ ]:
# 2. Cài dependency
!pip install -q -e ".[viz,eval]"


In [ ]:
# 3. Đăng nhập HuggingFace bằng Colab Secret 'HF_TOKEN'
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
os.environ['HF_TOKEN'] = HF_TOKEN
print('Đã đăng nhập HuggingFace.')


In [ ]:
# 4. Clone 3DINO (chứa package `dinov2`) và tải weights
#    License CC BY-NC-ND: clone nguyên gốc, KHÔNG sửa code trong vendor/.
import glob, shutil
from huggingface_hub import snapshot_download

REPO = os.path.join(PROJECT, 'vendor', '3DINO')
if not os.path.isdir(REPO):
    !git clone -q https://github.com/AICONSlab/3DINO {REPO}
!pip install -q -r {REPO}/requirements.txt

ARTIFACTS = os.path.join(PROJECT, 'artifacts')
os.makedirs(ARTIFACTS, exist_ok=True)
DINO_WEIGHTS = os.path.join(ARTIFACTS, '3dino_vit_weights.pth')
if not os.path.exists(DINO_WEIGHTS):
    downloaded = snapshot_download('AICONSlab/3DINO-ViT', token=HF_TOKEN)
    shutil.copy(glob.glob(os.path.join(downloaded, '3dino_vit_weights.pth'))[0], DINO_WEIGHTS)
print('3DINO code   :', REPO)
print('3DINO weights:', DINO_WEIGHTS)


## Cấu hình

Cấu hình là dataclass bất biến nạp từ YAML. Ghi đè ở đây **có tác dụng thật** vì
`cfg` được truyền tường minh xuống mọi hàm — khác với cách dùng hằng số cấp
module, nơi `from module import CONSTANT` sao chép giá trị lúc import và mọi ghi
đè sau đó bị nuốt im lặng.


In [ ]:
# 5. Nạp cấu hình
import logging
from knee_mri.config import load_config
from knee_mri.utils.logging import setup_logging
from knee_mri.utils.seed import set_seed

cfg = load_config('colab', overrides={
    'paths': {
        'data_root': os.path.join(PROJECT, 'dataset'),
        'artifacts_root': ARTIFACTS,
        'dino_code_dir': REPO,
    },
})
cfg.paths.ensure()
setup_logging(logging.INFO, log_dir=cfg.paths.logs)
set_seed(cfg.seed)

import torch
print('dataset      :', cfg.paths.data_root)
print('target_shape :', cfg.data.target_shape)
print('VLM          :', cfg.vlm.model_id)
print('thiết bị     :', 'cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
# 6. Nạp catalog (index CSV MỘT LẦN, tra cứu O(1) sau đó)
from knee_mri.data.catalog import StudyCatalog

catalog = StudyCatalog.from_csv(cfg.paths.train_csv, cfg.paths.train_series_csv)
study_uids = catalog.studies_with_series()

# Bắt đầu với tập nhỏ để kiểm tra luồng; đặt LIMIT = None để chạy full.
LIMIT = 50
subset = study_uids[:LIMIT] if LIMIT else study_uids
print(catalog, '| số study sẽ xử lý:', len(subset))


## S1 — Mask ROI và weak label

In [ ]:
# 7. Sinh mask ROI bằng SAM
from knee_mri.pipeline import generate_masks

generate_masks(cfg, catalog, subset, device='cuda')


In [ ]:
# 8. Sinh weak label từ report
#    use_vlm=True dùng VLM đọc hiểu; False dùng luật từ khóa đa ngôn ngữ (CPU).
from knee_mri.pipeline import precompute_weak_labels

precompute_weak_labels(cfg, catalog, subset, use_vlm=True, device='cuda')


## S2 — Precompute teacher rồi huấn luyện student

In [ ]:
# 9. Đặc trưng ảnh từ 3DINO-ViT (teacher đóng băng → cache một lần)
from knee_mri.pipeline import precompute_teacher_features

precompute_teacher_features(cfg, catalog, subset, device='cuda')


In [ ]:
# 10. Guidance từ VLM đa modality (ảnh + report → G)
from knee_mri.pipeline import precompute_guidance

precompute_guidance(cfg, catalog, subset, device='cuda')


In [ ]:
# 11. Kiểm tra cache trước khi huấn luyện
from knee_mri.data.dataset import describe_cache

print(describe_cache(cfg, subset))


In [ ]:
# 12. Huấn luyện student — 58 study có nhãn gold được giữ làm validation
from knee_mri.data.dataset import KneeDataset
from knee_mri.data.splits import make_split
from knee_mri.training.trainer import train

split = make_split(catalog, seed=cfg.seed, limit=LIMIT)
train_ds = KneeDataset(split.train, catalog, cfg)
val_ds = KneeDataset(split.val, catalog, cfg) if split.val else None

result = train(cfg, train_ds, val_ds, device='cuda')
print('Macro AUC tốt nhất:', result.best_metric)
print('Checkpoint        :', result.best_checkpoint)


## S3 — Giải thích và sinh submission

In [ ]:
# 13. CAM + báo cáo, đối chiếu với mask S1
from knee_mri.pipeline import explain_studies

results = explain_studies(cfg, catalog, subset[:20], use_vlm=True, device='cuda')
for item in results[:3]:
    print(item.study_uid[:24], '| IoU CAM/ROI:', item.overlap)


In [ ]:
# 14. Sinh submission.csv (CHỈ ảnh + metadata)
from knee_mri.inference import predict_submission

submission_path = predict_submission(cfg, device='cuda')
print('Submission:', submission_path)

import pandas as pd
pd.read_csv(submission_path).head()


## Tiếp theo

Mở [`03_results_visualization.ipynb`](03_results_visualization.ipynb) để xem
phân bố dự đoán, CAM overlay và đường ROC trên tập nhãn gold.
